
# Algoritmia · Práctica 5
## Algoritmos voraces: decisiones locales, hipótesis y contraejemplos

Hoy la práctica será distinta.

No vas a seguir un algoritmo explicado paso a paso. Vas a **proponer reglas**, programarlas, intentar romperlas y comparar sus resultados.

La pregunta que guiará toda la sesión es:

\[
\boxed{\text{¿Una buena decisión ahora conduce necesariamente a una buena solución final?}}
\]

Trabaja preferiblemente **por parejas**.

### Objetivos

Al terminar la práctica deberías ser capaz de:

- reconocer la estructura de un algoritmo voraz;
- formular una regla de elección local;
- construir contraejemplos;
- distinguir entre «funciona en mis pruebas» y «siempre es correcto»;
- analizar el coste de una solución voraz sencilla.

### Regla de la sesión

Antes de preguntar al profesor, prueba una de estas tres cosas:

1. ejecuta tu algoritmo con otro caso;
2. dibuja el problema en papel;
3. intenta construir un caso pequeño que contradiga tu hipótesis.



# 0. Preparación

Ejecuta esta celda. Solo carga herramientas que utilizaremos más adelante.


In [ ]:
from itertools import combinations
import random

random.seed(7)



# 1. Misión 1 · Diseña una estrategia para dar cambio

Disponemos de monedas de determinados valores y queremos devolver una cantidad utilizando **el menor número posible de monedas**.

Por ejemplo:

```python
monedas = [50, 20, 10, 5, 2, 1]
cantidad = 87
```

Antes de programar, escribe vuestra regla:

> **Nuestra estrategia será:**  
> ................................................................................  
> ................................................................................

Ahora implementadla.

La función debe devolver una lista con las monedas utilizadas.

Ejemplo posible:

```python
[50, 20, 10, 5, 2]
```


In [ ]:
def cambio_voraz(monedas, cantidad):
    # TODO
    # Implementa vuestra regla voraz.
    #
    # Recomendación:
    # - no modifiques la lista original;
    # - devuelve una lista con las monedas elegidas.

    pass



## 1.1. Primera prueba

Ejecuta vuestra función con:


In [ ]:
monedas = [50, 20, 10, 5, 2, 1]
cantidad = 87

solucion = cambio_voraz(monedas, cantidad)

print("Solución:", solucion)

if solucion is not None:
    print("Suma:", sum(solucion))
    print("Número de monedas:", len(solucion))



### Antes de continuar

¿Creéis que vuestra regla **siempre** devuelve el menor número posible de monedas?

- [ ] Sí
- [ ] No
- [ ] No lo sabemos todavía

Justificación breve:

................................................................................  
................................................................................



# 2. Misión 2 · Intenta romper tu algoritmo

Ahora vamos a comparar vuestra solución con una solución óptima en cantidades pequeñas.

La siguiente función actúa como **verificador**. No tenéis que modificarla ni estudiar su implementación.

Su única finalidad es decirnos cuántas monedas necesita una solución óptima.


In [ ]:
def minimo_monedas_verificador(monedas, cantidad):
    monedas = sorted(monedas, reverse=True)
    mejor = None

    def buscar(restante, usadas, inicio):
        nonlocal mejor

        if restante == 0:
            if mejor is None or len(usadas) < len(mejor):
                mejor = usadas[:]
            return

        if restante < 0:
            return

        if mejor is not None and len(usadas) >= len(mejor):
            return

        for i in range(inicio, len(monedas)):
            moneda = monedas[i]
            usadas.append(moneda)
            buscar(restante - moneda, usadas, i)
            usadas.pop()

    buscar(cantidad, [], 0)
    return mejor



## 2.1. Prueba estos casos

Para cada caso compara:

- vuestra solución;
- el número de monedas utilizado;
- la solución óptima del verificador.

¿Aparece algún fallo?


In [ ]:
casos = [
    ([50, 20, 10, 5, 2, 1], 87),
    ([10, 5, 1], 28),
    ([4, 3, 1], 6),
    ([6, 4, 1], 8),
]

for monedas, cantidad in casos:
    voraz = cambio_voraz(monedas, cantidad)
    optima = minimo_monedas_verificador(monedas, cantidad)

    print("=" * 55)
    print("Monedas:", monedas, "Cantidad:", cantidad)
    print("Voraz: ", voraz, "->", None if voraz is None else len(voraz), "monedas")
    print("Óptima:", optima, "->", None if optima is None else len(optima), "monedas")



## 2.2. Cazadores de contraejemplos

Ahora no vale utilizar los casos anteriores.

Construid vosotros un sistema de monedas y una cantidad para los que vuestra estrategia voraz **no produzca una solución óptima**.

Intentad que el ejemplo sea lo más pequeño posible.

```text
Monedas:
Cantidad:
Solución voraz:
Solución óptima:
```

Después comprobadlo con el verificador.


In [ ]:
# TODO: escribe vuestro contraejemplo

mis_monedas = []
mi_cantidad = 0

# voraz = cambio_voraz(mis_monedas, mi_cantidad)
# optima = minimo_monedas_verificador(mis_monedas, mi_cantidad)
#
# print("Voraz:", voraz)
# print("Óptima:", optima)



### Conclusión provisional

Completa:

> Una estrategia voraz puede parecer razonable y funcionar en muchos casos, pero  
> ................................................................................  
> ................................................................................

\[
\boxed{\text{Una elección local razonable no garantiza por sí sola una solución global óptima}}
\]



# 3. Misión 3 · Un nuevo problema

Tenemos varias actividades. Cada actividad tiene:

```text
(nombre, inicio, fin)
```

Por ejemplo:


In [ ]:
actividades = [
    ("A", 1, 4),
    ("B", 3, 5),
    ("C", 0, 6),
    ("D", 5, 7),
    ("E", 3, 9),
    ("F", 5, 9),
    ("G", 6, 10),
    ("H", 8, 11),
    ("I", 8, 12),
    ("J", 2, 14),
    ("K", 12, 16),
]

for actividad in actividades:
    print(actividad)



Dos actividades son compatibles si una puede empezar cuando la otra ya ha terminado.

Por ejemplo:

```text
A = (1, 4)
D = (5, 7)
```

son compatibles.

Pero:

```text
A = (1, 4)
B = (3, 5)
```

se solapan.

## Objetivo

Queremos seleccionar **el mayor número posible de actividades compatibles**.

### Antes de programar: proponed una regla

Elegid una de estas hipótesis:

**H1.** Elegir primero la actividad que **empieza antes**.

**H2.** Elegir primero la actividad de **menor duración**.

**H3.** Elegir primero la actividad que **termina antes**.

También podéis proponer vuestra propia regla.

> **Nuestra hipótesis:** ............................................................  
> **Creemos que puede funcionar porque:** ...........................................  
> ................................................................................



# 4. Misión 4 · Implementa vuestra regla

La función debe devolver una lista de actividades compatibles.

No hay una única implementación posible.

Podéis ordenar primero las actividades según vuestro criterio y después recorrerlas.


In [ ]:
def seleccionar_actividades(actividades):
    # TODO
    # Implementa vuestra estrategia.

    pass



## 4.1. Pruébala


In [ ]:
seleccionadas = seleccionar_actividades(actividades)

print("Seleccionadas:")
print(seleccionadas)

if seleccionadas is not None:
    print("Número de actividades:", len(seleccionadas))



### ¿Estáis seguros de que es óptima?

No respondáis todavía mirando solo este ejemplo.

Primero intentad construir **otro conjunto de actividades** en el que vuestra regla pueda comportarse mal.


In [ ]:
# TODO: crea vuestro propio caso de prueba

actividades_prueba = [
    # ("A", inicio, fin),
]

# print(seleccionar_actividades(actividades_prueba))



# 5. Misión 5 · Un árbitro para nuestras hipótesis

La siguiente función prueba muchas combinaciones y obtiene una solución óptima para conjuntos pequeños.

**No tenéis que modificarla.** La usaremos únicamente como árbitro.


In [ ]:
def son_compatibles(conjunto):
    ordenadas = sorted(conjunto, key=lambda x: x[1])

    for i in range(len(ordenadas) - 1):
        if ordenadas[i][2] > ordenadas[i + 1][1]:
            return False

    return True


def optimo_actividades(actividades):
    mejor = []

    for r in range(1, len(actividades) + 1):
        for subconjunto in combinations(actividades, r):
            if son_compatibles(subconjunto) and len(subconjunto) > len(mejor):
                mejor = list(subconjunto)

    return mejor



Compara vuestra solución con la óptima:


In [ ]:
voraz = seleccionar_actividades(actividades)
optima = optimo_actividades(actividades)

print("Voraz:", voraz)
print("Tamaño voraz:", None if voraz is None else len(voraz))

print()

print("Óptima:", optima)
print("Tamaño óptimo:", len(optima))



Si coinciden en tamaño, todavía no hemos demostrado nada.

Solo sabemos que nuestra regla ha funcionado **en este caso**.



# 6. Misión 6 · Somete tu hipótesis a estrés

Vamos a generar pequeños conjuntos de actividades aleatorias.

El verificador encontrará el óptimo y compararemos vuestra estrategia con él.

Primero ejecuta el generador.


In [ ]:
def generar_actividades(n, inicio_max=15, duracion_max=7):
    resultado = []

    for i in range(n):
        inicio = random.randint(0, inicio_max)
        duracion = random.randint(1, duracion_max)
        fin = inicio + duracion

        resultado.append((f"A{i}", inicio, fin))

    return resultado



Ahora ejecutaremos vuestra estrategia sobre varios casos.

Si aparece un fallo, guardaremos el primer contraejemplo encontrado.


In [ ]:
def probar_estrategia(estrategia, repeticiones=50, n=7):
    aciertos = 0
    primer_fallo = None

    for _ in range(repeticiones):
        caso = generar_actividades(n)

        solucion = estrategia(caso)
        optima = optimo_actividades(caso)

        if solucion is not None and len(solucion) == len(optima):
            aciertos += 1
        elif primer_fallo is None:
            primer_fallo = (caso, solucion, optima)

    return aciertos, primer_fallo


In [ ]:
aciertos, fallo = probar_estrategia(
    seleccionar_actividades,
    repeticiones=50,
    n=7
)

print("Casos resueltos con tamaño óptimo:", aciertos, "/ 50")

if fallo is None:
    print("No hemos encontrado un contraejemplo.")
else:
    caso, voraz, optima = fallo

    print("\nPrimer contraejemplo encontrado:")
    print("Caso:", caso)
    print("Voraz:", voraz)
    print("Óptima:", optima)



## Interpreta el experimento

Responded:

1. ¿Cuántos de los 50 casos resolvió óptimamente vuestra estrategia?

   **Respuesta:** ...................................................

2. Si ha fallado, ¿qué característica tiene el contraejemplo?

   ................................................................................  
   ................................................................................

3. Si no ha fallado, ¿podemos afirmar que siempre funcionará?

   ................................................................................  
   ................................................................................

4. ¿Qué diferencia existe entre **no encontrar un contraejemplo** y **demostrar que no existe**?

   ................................................................................  
   ................................................................................



# 7. Misión 7 · ¿Qué regla parece especial?

Ahora vamos a comparar directamente tres estrategias.

Debéis implementar las tres:

- empieza antes;
- menor duración;
- termina antes.

Intentad reutilizar código.


In [ ]:
def voraz_empieza_antes(actividades):
    # TODO
    pass


def voraz_menor_duracion(actividades):
    # TODO
    pass


def voraz_termina_antes(actividades):
    # TODO
    pass



Somételas al mismo banco de pruebas:


In [ ]:
estrategias = {
    "Empieza antes": voraz_empieza_antes,
    "Menor duración": voraz_menor_duracion,
    "Termina antes": voraz_termina_antes,
}

for nombre, estrategia in estrategias.items():
    aciertos, fallo = probar_estrategia(
        estrategia,
        repeticiones=100,
        n=7
    )

    print(f"{nombre:18s}: {aciertos}/100 soluciones óptimas")



## Discusión

Completa la tabla con vuestros resultados:

| Regla | Aciertos / 100 | ¿Encontramos contraejemplo? |
|---|---:|:---:|
| Empieza antes | | |
| Menor duración | | |
| Termina antes | | |

### Pregunta

¿Cuál de las tres reglas parece comportarse de forma diferente?

................................................................................

¿Los experimentos demuestran que esa regla siempre es correcta?

................................................................................



# 8. Misión 8 · Destruye las reglas equivocadas

Para cada estrategia que haya fallado, intentad construir **a mano** un contraejemplo pequeño.

No utilicéis el generador aleatorio.

La intención no es buscar muchos casos. Es entender **por qué** la regla falla.


In [ ]:
contraejemplo_inicio = [
    # TODO
]

contraejemplo_duracion = [
    # TODO
]



# 9. Misión 9 · Construye el voraz que mejor ha resistido

A estas alturas deberíais tener una candidata clara.

Implementad una versión final y limpia:

```python
seleccion_voraz(actividades)
```

La función debe:

1. ordenar según vuestra regla;
2. recorrer las actividades una sola vez;
3. aceptar únicamente las compatibles;
4. devolver la selección.


In [ ]:
def seleccion_voraz(actividades):
    # TODO

    pass



Pruébala con el conjunto original:


In [ ]:
resultado = seleccion_voraz(actividades)

print(resultado)
print("Número de actividades:", None if resultado is None else len(resultado))



# 10. Misión 10 · ¿Cuánto trabajo estamos haciendo?

Supongamos que tenemos `n` actividades.

Vuestra solución final hace dos tareas principales:

### A. Ordenar las actividades

¿Qué coste tiene ordenar `n` elementos?

\[
O(\_\_\_\_\_\_\_\_)
\]

### B. Recorrer la lista ordenada

¿Cuántas actividades se inspeccionan?

\[
O(\_\_\_\_\_\_\_\_)
\]

### C. Coste total

Los dos bloques son secuenciales:

\[
T(n)=T_{\text{ordenación}}(n)+T_{\text{recorrido}}(n)
\]

Por tanto:

\[
\boxed{O(\_\_\_\_\_\_\_\_)}
\]

Explica cuál es el término dominante:

................................................................................



# 11. ¿Qué hemos descubierto?

Volvamos a las monedas y a las actividades.

En ambos problemas hicimos algo parecido:

```text
1. ordenar o priorizar candidatos
2. elegir el que parece mejor ahora
3. incorporarlo si es válido
4. continuar
5. no volver atrás
```

Eso es la idea básica de un **algoritmo voraz**.

Pero las dos experiencias han sido distintas:

### Cambio de monedas

Una regla muy natural puede fallar.

### Selección de actividades

Una determinada regla parece mucho más robusta que las demás.

Por tanto:

\[
\boxed{\text{ser voraz es una estrategia de diseño, no una garantía de optimalidad}}
\]



# 12. Entrega final · Cinco respuestas

Antes de terminar, cada pareja debe poder responder claramente a estas cinco preguntas.

### 1. ¿Qué caracteriza a un algoritmo voraz?

................................................................................  
................................................................................

### 2. ¿Qué es una elección local?

................................................................................  
................................................................................

### 3. ¿Por qué un algoritmo voraz puede fallar aunque su regla parezca razonable?

................................................................................  
................................................................................

### 4. ¿Qué diferencia hay entre probar muchos ejemplos y demostrar que un algoritmo siempre funciona?

................................................................................  
................................................................................

### 5. ¿Qué regla ha funcionado mejor en el problema de selección de actividades?

................................................................................



# Para quedarte con una sola idea

\[
\boxed{
\text{proponer una elección local}
\rightarrow
\text{probar}
\rightarrow
\text{buscar contraejemplos}
\rightarrow
\text{justificar}
}
\]

La semana que viene utilizaremos esta misma forma de pensar en un problema nuevo: **grafos y caminos mínimos**.
